In [2]:
import pandas as pd
import numpy as np

## 1. Caricamento del dataset `manuale.csv`

In questa sezione carichiamo il file `manuale.csv`, che contiene un sottoinsieme di 14 campioni
estratti dal dataset originale e bilanciati rispetto alla variabile target `Diabetes_binary`.

In [5]:
# Caricamento del file manuale.csv

manual_df = pd.read_csv("../data/manuale.csv")

manual_df.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,0,1,21,0,0,0,1,0,1,...,0,3,3,2,0,0,2,6,8,0
1,0,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,13,5,6,0
2,1,1,1,33,1,0,0,1,0,1,...,0,3,0,0,0,1,8,4,8,1
3,0,0,1,23,1,0,0,0,1,1,...,0,3,0,0,0,0,5,3,5,0
4,0,1,1,37,0,0,0,0,1,1,...,0,4,20,30,1,0,10,2,2,1


## 2. Separazione tra feature e target

- `X_manual`: contiene tutte le feature esplicative.
- `y_manual`: contiene la variabile target `Diabetes_binary`.

In [6]:
# Separiamo le feature dalla variabile target

X_manual = manual_df.drop(columns=["Diabetes_binary"])
y_manual = manual_df["Diabetes_binary"]

# Controlliamo le dimensioni
print("Shape X_manual:", X_manual.shape)
print("Shape y_manual:", y_manual.shape)

# Verifichiamo la distribuzione della classe
y_manual.value_counts()


Shape X_manual: (14, 21)
Shape y_manual: (14,)


Diabetes_binary
0    7
1    7
Name: count, dtype: int64

## 3. Richiamo teorico: Classificatore Naive Bayes

Il classificatore Naive Bayes si basa sul teorema di Bayes:

P(Y | X) = [ P(X | Y) P(Y) ] / P(X)
dove:
- Y rappresenta la variabile target,
- X rappresenta il vettore delle feature osservate.

Nel problema di classificazione, il termine P(X) è uguale per tutte le classi e può essere ignorato durante il confronto.

Pertanto il classificatore assegna una nuova osservazione alla classe che massimizza:

P(Y = y | X) ∝ P(Y = y) ∏i=1 P(X_i | Y = y)

L’ipotesi “Naive” consiste nel considerare le feature condizionalmente indipendenti dato  Y .

Nel nostro caso:
- Y in {0, 1}
- Le feature sono principalmente binarie o discrete, quindi possiamo usare una versione di Naive Bayes per variabili discrete.
- Per evitare problemi con probabilità nulle, utilizziamo il Laplace smoothing:
P(X_i = x | Y = y) = [ N(X_i = x, Y = y) + 1 ] / N(Y = y) + k
dove k  è il numero di valori possibili per la feature  X_i.


## 4. Analisi preliminare del dataset manuale

Prima di costruire il classificatore, analizziamo la distribuzione delle classi nel dataset `manuale.csv`.

Questo passaggio permette di calcolare le probabilità a priori delle classi e verificare il bilanciamento del dataset utilizzato per la costruzione manuale del classificatore.

In [7]:
# Distribuzione delle classi nel dataset manuale

class_distribution = y_manual.value_counts()

print(class_distribution)

Diabetes_binary
0    7
1    7
Name: count, dtype: int64


In [8]:
# Distribuzione percentuale delle classi

class_distribution_percent = y_manual.value_counts(normalize=True) * 100

print(class_distribution_percent)

Diabetes_binary
0    50.0
1    50.0
Name: proportion, dtype: float64


### Osservazioni

Il dataset `manuale.csv` contiene 14 osservazioni:

- 7 appartenenti alla classe 0 (assenza di diabete)
- 7 appartenenti alla classe 1 (presenza di diabete)

Il dataset risulta perfettamente bilanciato.

## 5. Calcolo delle probabilità a priori

Le probabilità a priori rappresentano la probabilità di osservare ciascuna classe prima di considerare le feature.

Sono calcolate come:

- numero di osservazioni della classe / numero totale di osservazioni

In [9]:
# Calcolo delle probabilità a priori

prior_0 = (y_manual == 0).mean()
prior_1 = (y_manual == 1).mean()

print(f"P(Y=0) = {prior_0:.4f}")
print(f"P(Y=1) = {prior_1:.4f}")

P(Y=0) = 0.5000
P(Y=1) = 0.5000


### Risultati

Le probabilità a priori risultano:

- P(Y=0) = 0.50
- P(Y=1) = 0.50

Poiché il dataset manuale è stato costruito in modo bilanciato, entrambe le classi hanno la stessa probabilità iniziale.

## 6. Selezione delle feature

Per la costruzione manuale del classificatore Naive Bayes sono state selezionate cinque feature considerate particolarmente rilevanti per la predizione del diabete:

- HighBP
- HighChol
- BMI
- Age
- PhysActivity


In [11]:
selected_features = [
    "HighBP",
    "HighChol",
    "BMI",
    "Age",
    "PhysActivity"
]

manual_df[selected_features + ["Diabetes_binary"]]

,HighBP,HighChol,BMI,Age,PhysActivity,Diabetes_binary
0,1,0,21,2,1,0
1,0,0,27,13,1,0
2,1,1,33,8,1,1
3,0,0,23,5,0,0
4,0,1,37,10,0,1
5,1,1,25,11,0,0
6,1,0,26,11,0,1
7,0,1,29,12,1,1
8,1,0,24,10,1,0
9,1,1,29,9,1,1


## 7. Calcolo delle probabilità condizionate

Per costruire il classificatore Naive Bayes è necessario stimare le probabilità condizionate delle feature rispetto alla classe.

Si inizia analizzando una variabile alla volta, costruendo una tabella di contingenza che mostra la distribuzione dei valori della feature nelle due classi della variabile target.

In [12]:
# Tabella di contingenza per HighBP

highbp_table = pd.crosstab(
    manual_df["HighBP"],
    manual_df["Diabetes_binary"]
)

highbp_table

Diabetes_binary,0,1
HighBP,,
0,4,3
1,3,4


### Probabilità condizionate per HighBP

La feature `HighBP` è binaria e può assumere due valori possibili (0 oppure 1).

Le probabilità condizionate ottenute sono:

### Classe Y = 0

- P(HighBP=0 | Y=0) = 4/7 = 0.571
- P(HighBP=1 | Y=0) = 3/7 = 0.428

### Classe Y = 1

- P(HighBP=0 | Y=1) = 3/7 = 0.428
- P(HighBP=1 | Y=1) = 4/7 = 0.571

In [ ]:
# Tabella di contingenza per HighChol

highChol_table = pd.crosstab(
    manual_df["HighChol"],
    manual_df["Diabetes_binary"]
)

highChol_table

Diabetes_binary,0,1
HighChol,,
0,6,2
1,1,5


### Probabilità condizionate per HighChol

La feature `HighChol` indica la presenza o l'assenza di colesterolo alto.

Le probabilità condizionate ottenute sono:

### Classe Y = 0

- P(HighChol = 0 | Y = 0) = 6/7 = 0.857
- P(HighChol = 1 | Y = 0) = 1/7 = 0.142

### Classe Y = 1

- P(HighChol = 0 | Y = 1) = 2/7 = 0.285
- P(HighChol = 1 | Y = 1) = 5/7 = 0.714

Si osserva che la presenza di colesterolo alto (`HighChol = 1`) è molto più frequente tra i soggetti diabetici rispetto ai non diabetici.

In [14]:
# Tabella di contingenza per BMI

BMI_table = pd.crosstab(
    manual_df["BMI"],
    manual_df["Diabetes_binary"]
)

BMI_table

Diabetes_binary,0,1
BMI,,
21,1,0
23,2,1
24,1,0
25,1,0
26,0,1
27,2,0
29,0,2
32,0,1
33,0,1


### Probabilità condizionate per BMI

La feature `BMI` è una variabile numerica discreta.

Si osserva che i valori più elevati di BMI risultano presenti prevalentemente nella classe dei soggetti diabetici.

In particolare:

- BMI pari a 29 compare esclusivamente nella classe 1;
- BMI pari a 32, 33 e 37 compaiono esclusivamente nella classe 1;
- BMI pari a 21, 24, 25 e 27 compaiono esclusivamente nella classe 0.

Questa distribuzione suggerisce una possibile relazione positiva tra valori elevati di BMI e presenza del diabete.

Tuttavia, il numero ridotto di osservazioni presenti nel dataset `manuale.csv` non consente di trarre conclusioni statisticamente robuste. 

Per questo motivo è stata effettuata una discretizzazione della variabile BMI in tre categorie:

- BMI <= 24        ===> Normal
- 25 <= BMI <= 29  ===> Overweight
- BMI >= 30        ===> Obese


Questa trasformazione riduce la frammentazione dei dati e rende più robusta la stima delle probabilità condizionate utilizzate dal classificatore Naive Bayes.

In [15]:
# Discretizzazione del BMI

manual_df["BMI_Category"] = pd.cut(
    manual_df["BMI"],
    bins=[0, 25, 30, 100],
    labels=["Normal", "Overweight", "Obese"],
    right=False
)

manual_df[["BMI", "BMI_Category"]]

,BMI,BMI_Category
0,21,Normal
1,27,Overweight
2,33,Obese
3,23,Normal
4,37,Obese
5,25,Overweight
6,26,Overweight
7,29,Overweight
8,24,Normal
9,29,Overweight


In [ ]:
# Tabella di contingenza per BMI_Categorry

pd.crosstab(
    manual_df["BMI_Category"],
    manual_df["Diabetes_binary"]
)

Diabetes_binary,0,1
BMI_Category,,
Normal,4,1
Overweight,3,3
Obese,0,3


### Probabilità condizionate per BMI_Category

Poiché la categoria `Obese` non compare nella classe 0, si verifica una probabilità nulla.

Per evitare che una singola probabilità nulla annulli l'intera probabilità della classe durante il prodotto delle probabilità condizionate, viene applicato il **Laplace Smoothing**.

Essendo presenti tre categorie possibili (`k = 3`), le probabilità condizionate ottenute sono:

### Classe Y = 0

- P(Normal | Y = 0) = (4 + 1) / (7 + 3) = 5/10 = 0.50
- P(Overweight | Y = 0) = (3 + 1) / (7 + 3) = 4/10 = 0.40
- P(Obese | Y = 0) = (0 + 1) / (7 + 3) = 1/10 = 0.10

### Classe Y = 1

- P(Normal | Y = 1) = (1 + 1) / (7 + 3) = 2/10 = 0.20
- P(Overweight | Y = 1) = (3 + 1) / (7 + 3) = 4/10 = 0.40
- P(Obese | Y = 1) = (3 + 1) / (7 + 3) = 4/10 = 0.40

Dall'analisi emerge che la categoria `Obese` è maggiormente associata alla presenza di diabete, mentre la categoria `Normal` è più frequente nei soggetti non diabetici. 

In [10]:
# Tabella di contingenza per Age

Age_table = pd.crosstab(
    manual_df["Age"],
    manual_df["Diabetes_binary"]
)

Age_table

Diabetes_binary,0,1
Age,,
2,1,0
4,1,0
5,1,0
8,0,1
9,0,1
10,1,3
11,1,1
12,0,1
13,2,0


### Calcolo delle probabilità condizionate per la feature Age

La feature Age è un attributo ordinale che rappresenta fasce d’età codificate come valori interi.
Tuttavia, questi valori presentano molte categorie distinte, alcune delle quali compaiono una sola volta o addirittura non compaiono affatto in una delle due classi.

Per rendere il modello più stabile e coerente, è opportuno discretizzare Age in 4 categorie più ampie, che rappresentano fasce d’età significative:

- Age <= 4         ===> Bambino
- 5 <= Age <= 8    ===> Giovane
- 9 <= Age <= 11   ===> Adulto
- Age >= 12        ===> Anziano

Questa trasformazione riduce la frammentazione dei dati e rende più robusta la stima delle probabilità condizionate utilizzate dal classificatore Naive Bayes.

In [11]:
# Discretizzazione del Age

manual_df["Age_Category"] = pd.cut(
    manual_df["Age"],
    bins=[0, 5, 9, 12, 14],
    labels=["Bambino", "Giovane", "Adulto", "Anziano"],
    right=False
)

manual_df[["Age", "Age_Category"]]


,Age,Age_Category
0,2,Bambino
1,13,Anziano
2,8,Giovane
3,5,Giovane
4,10,Adulto
5,11,Adulto
6,11,Adulto
7,12,Anziano
8,10,Adulto
9,9,Adulto


In [12]:
# Tabella di contingenza per Age_Category
pd.crosstab(
    manual_df["Age_Category"],
    manual_df["Diabetes_binary"]
)


Diabetes_binary,0,1
Age_Category,,
Bambino,2,0
Giovane,1,1
Adulto,2,5
Anziano,2,1


### Probabilità condizionate per Age_Category 

Poiché la categoria `Bambino` non compare nella classe 0, si verifica una probabilità nulla.

Per evitare che una singola probabilità nulla annulli l'intera probabilità della classe durante il prodotto delle probabilità condizionate, viene applicato il **Laplace Smoothing**.

Essendo presenti quattro categorie possibili (`k = 4`), le probabilità condizionate ottenute sono:

### Classe Y = 0

- P(Bambino | Y=0) = (2 + 1) / 11 = 3 / 11 = 0.273  
- P(Giovane | Y=0) = (1 + 1) / 11 = 2 / 11 = 0.182  
- P(Adulto | Y=0) = (2 + 1) / 11 = 3 / 11 = 0.273  
- P(Anziano | Y=0) = (2 + 1) / 11 = 3 / 11 = 0.273  


### Classe Y = 1

- P(Bambino | Y=1) = (0 + 1) / 11 = 1 / 11 = 0.091  
- P(Giovane | Y=1) = (1 + 1) / 11 = 2 / 11 = 0.182  
- P(Adulto | Y=1) = (5 + 1) / 11 = 6 / 11 = 0.545  
- P(Anziano | Y=1) = (1 + 1) / 11 = 2 / 11 = 0.182 


Dall'analisi emerge che la categoria `Adulto` è maggiormente associata alla presenza di diabete


In [14]:
# Tabella di contingenza per physActivity

PhysActivity_table = pd.crosstab(
    manual_df["PhysActivity"],
    manual_df["Diabetes_binary"]
)

PhysActivity_table

Diabetes_binary,0,1
PhysActivity,,
0,3,2
1,4,5


### Calcolo delle probabilità condizionate per PhysActivity

La feature **PhysActivity** è binaria e indica se il soggetto ha svolto attività fisica negli ultimi 30 giorni (0 = No, 1 = Sì).

Le probabilità condizionate ottenute sono:

### Classe Y = 0

- P(PhysActivity = 0 | Y=0) = 3 / 7 = 0.429  
- P(PhysActivity = 1 | Y=0) = 4 / 7 = 0.571  

### Classe Y = 1

- P(PhysActivity = 0 | Y=1) = 2 / 7 = 0.286  
- P(PhysActivity = 1 | Y=1) = 5 / 7 = 0.714  

Osservazione:  
La classe Y=1 (diabetici) mostra una proporzione maggiore di soggetti che dichiarano attività fisica (5 su 7).  
Questo potrebbe indicare che l’attività fisica non è sufficiente da sola a discriminare la classe, ma contribuisce comunque al modello Naive Bayes.
